In [ ]:
import pandas as pd
# Load input CSV
df = pd.read_csv("ADDRESS")
df

In [ ]:
import openai

import time

# Set your OpenAI API key
myKey = "YOUR_KEY"

client = openai.OpenAI(api_key=myKey)


In [ ]:
import csv
from tqdm import tqdm
# Define output CSV and write header
output_file = "ADDRESS"
with open(output_file, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        'qid', 'context', 'label','original_question', 'suggested_question','gpt_answer'
    ])

# Iterate over rows
for i, row in tqdm(df.iterrows(), total=len(df)):
    qid = row['qid']
    context = row['context']
    label = row['label']
    uq = row['UQ_question']
    sq = row['SA_question']
    # print(f"Processing row {i}: {qid}")

    prompt = f"""You are a question answering model. Your task is to determine whether each of the following questions is answerable using only the information provided in the context.

    Instructions:
    - Only output the answers in the format shown below.
    - Do NOT repeat the questions.
    - If the context contains the answer, extract it exactly as it appears in the context (copy the exact span). Do not use any external knowledge.
    - If the context does not contain the answer, respond with exactly: Null

    ---

    Context:
    {context}

    Question 1:
    {uq}

    Question 2:
    {sq}

    Respond strictly in this format:

    Answer 1: <your answer here>
    Answer 2: <your answer here>

    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        gpt_answer = response.choices[0].message.content.strip()
        # print(f"Response: {content}")

    except Exception as e:
        print(f"Error on row {i}: {e}")
        gpt_answer = e


    # Append row to output file
    with open(output_file, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([qid, context,label, uq, sq, gpt_answer])

    # time.sleep(1)

In [ ]:
import pandas as pd
import re

def split_gpt_answers(df, answer_col="gpt_answer"):
    """
    Splits the 'gpt_answer' column into 'answer_1' and 'answer_2' columns.
    Assumes format:
    Answer 1: ...
    Answer 2: ...
    """
    answer1_list = []
    answer2_list = []

    for text in df[answer_col]:
        # Ensure text is a string
        text = str(text)

        # Extract answers using regex
        match1 = re.search(r"Answer\s*1\s*:\s*(.*?)(?:\n|$)", text)
        match2 = re.search(r"Answer\s*2\s*:\s*(.*?)(?:\n|$)", text)

        a1 = match1.group(1).strip() if match1 else "Null"
        a2 = match2.group(1).strip() if match2 else "Null"

        answer1_list.append(a1)
        answer2_list.append(a2)

    # Add to dataframe
    df["original_question_gpt_answer"] = answer1_list
    df["suggested_question_gpt_answer"] = answer2_list

    return df

df_new = pd.read_csv("ADDRESS")
df_new = split_gpt_answers(df_new)
df_new.to_csv("ADDRESS", index=False)
df_new
